# 🎵 《飞鸟集·两种飞行》ACE-Step 1.5 AI 音乐生成引擎 (Colab A100 SXM4 40GB 纯净版)
> **专为 Antigravity 自动化调度设计的无头 (Headless) 与交互式音乐引擎**
- **正式生产规格**：ACE-Step 1.5 XL SFT (4B DiT) + 5Hz LM (4B) · 50 Steps · CFG 7.0 · shift 3.0 · 48kHz 立体声
- **试听快速规格**：ACE-Step 1.5 XL Turbo (4B DiT) + 5Hz LM (4B) · 8 Steps · 无 CFG · shift 3.0
- **全自动生命周期**：无需人工 Gradio，支持同一 Kernel 持续复用模型生成，生成完成自动归档 Google Drive 并释放算力。

## 1. 硬件检测与算子环境变量配置

In [ ]:
import os, sys, time
# 关键：关闭 PyTorch 2.14+ 实验性 native Triton JIT，平滑回退至 NVIDIA cuBLAS 稳定算子，避免 RoPE 缺失 Triton kernel 错误
os.environ["TORCH_DISABLE_NATIVE_JIT"] = "1"
os.environ["ACESTEP_DISABLE_QUANTIZATION"] = "1"

import torch
assert torch.cuda.is_available(), "❌ 未检测到 GPU！请在 Colab 运行时中分配 A100 GPU。"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"✅ 算力就绪: {gpu_name} ({vram_gb:.1f} GiB VRAM) | PyTorch {torch.__version__}")
assert vram_gb >= 20, f"XL SFT/Turbo 4B 模型建议使用 >=20GB VRAM (当前: {vram_gb:.1f}GB)，请使用 A100 40GB/80GB。"

## 2. 代码库拉取与依赖环境配置 (去除 torchao 量化冲突)

In [ ]:
REPO = "/content/ACE-Step-1.5"
if not os.path.isdir(REPO):
    !git clone -q https://github.com/ACE-Step/ACE-Step-1.5.git {REPO}
    !cd {REPO} && git checkout -q ca1e85f || true

# 解耦训练集依赖，防止循环导入
train_init = f"{REPO}/acestep/training/__init__.py"
if os.path.isfile(train_init):
    txt = open(train_init, "r", encoding="utf-8").read()
    target = "from acestep.training.dataset_builder import DatasetBuilder, AudioSample"
    if target in txt:
        txt = txt.replace(target, "try:\n    from acestep.training.dataset_builder import DatasetBuilder, AudioSample\nexcept Exception:\n    DatasetBuilder, AudioSample = None, None")
        open(train_init, "w", encoding="utf-8").write(txt)

# 安装核心推理依赖（剔除并在 A100 上卸载 torchao 以保证 bfloat16 原生精度）
deps = ("transformers>=4.51.0,<4.58.0 diffusers>=0.37.0 accelerate>=1.12.0 einops loguru "
        "soundfile scipy numba>=0.63.1 vector-quantize-pytorch>=1.27.15 torchcodec>=0.9.1 "
        "toml modelscope diskcache safetensors==0.7.0 matplotlib typer-slim>=0.21.1 "
        "pytorch-wavelets pywavelets peft lycoris-lora lightning tensorboard "
        "fastapi uvicorn python-multipart nvidia-cuda-runtime-cu12")
!pip install -q {deps}
!pip uninstall -y -q torchao || true
!pip install -q --no-deps -e {REPO}
!pip install -q --no-deps -e {REPO}/acestep/third_parts/nano-vllm

# 预加载 nvidia 动态库
import ctypes, glob
for so in sorted(glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib/*.so*")):
    try:
        ctypes.CDLL(so, mode=ctypes.RTLD_GLOBAL)
    except Exception:
        pass

sys.path.insert(0, REPO)
os.chdir(REPO)
print("✅ 依赖与路径初始化完成。")

## 3. 权重拉取与模型加载驻留 (DiT 4B + 5Hz LM 4B)

In [ ]:
from pathlib import Path
from acestep.model_downloader import download_main_model, download_submodel
from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler

ckpt = Path(REPO) / "checkpoints"
ACE_MODEL = "acestep-v15-xl-sft"
ACE_LM = "acestep-5Hz-lm-4B"

print("📥 下载权重...")
ok, msg = download_main_model(checkpoints_dir=ckpt)
assert ok, msg
ok, msg = download_submodel(ACE_MODEL, checkpoints_dir=ckpt)
assert ok, msg
ok, msg = download_submodel(ACE_LM, checkpoints_dir=ckpt)
assert ok, msg

print("🚀 加载 DiT 音乐生成服务到显存...")
dit = AceStepHandler()
status, ok = dit.initialize_service(project_root=REPO, config_path=ACE_MODEL, device="cuda", offload_to_cpu=False, offload_dit_to_cpu=False)
assert ok, status

print("🧠 加载 5Hz LM 语言模型服务...")
llm = LLMHandler()
lm_backend = "vllm"
try:
    status, ok = llm.initialize(checkpoint_dir=str(ckpt), lm_model_path=ACE_LM, backend=lm_backend, device="cuda")
except Exception as e:
    lm_backend = "pt"
    status, ok = llm.initialize(checkpoint_dir=str(ckpt), lm_model_path=ACE_LM, backend=lm_backend, device="cuda")
assert ok, status

print(f"🎉 双引擎就绪！DiT: {ACE_MODEL} | LM: {ACE_LM} (backend={lm_backend})")

## 4. 音乐推理执行

In [ ]:
import soundfile as sf
from acestep.inference import GenerationParams, GenerationConfig, generate_music
import IPython.display as ipd

OUT = "/content/ace_outputs"
os.makedirs(OUT, exist_ok=True)

# 提示词与编曲参数
caption = "oriental, East Asian, watercolor atmosphere, animation soundtrack, gentle, clear, pure, poetic, soft piano, airy flute, gentle soft strings, slow tempo, no drums, evoking early spring, dawn morning, tree branch, a small bird, pure instrumental, high fidelity"
params = GenerationParams(
    task_type="text2music",
    caption=caption,
    lyrics="[Instrumental]",
    instrumental=True,
    duration=30.0,
    inference_steps=50,
    guidance_scale=7.0,
    shift=3.0,
    seed=42,
    bpm=72,
    keyscale="D Major",
    timesignature="4",
    thinking=True,
    use_cot_metas=True,
    use_cot_caption=True,
    use_cot_language=True
)
cfg = GenerationConfig(batch_size=1, use_random_seed=False, seeds=[42], audio_format="wav")

print("🎵 正在生成音乐...")
t0 = time.time()
res = generate_music(dit, llm, params, cfg, save_dir=OUT)
cost = time.time() - t0
assert res.success and res.audios, f"生成失败: {res.error}"

audio_path = res.audios[0]["path"]
info = sf.info(audio_path)
print(f"✅ 生成成功！耗时: {cost:.2f}s | 时长: {info.duration:.2f}s | 采样率: {info.samplerate}Hz | 声道: {info.channels}")

ipd.Audio(audio_path)